# Tin tức (Unstructured Data)

Notebook điều khiển riêng **luồng tin tức** (vnstock, RSS, HTML list).

| Đầu vào | Mô tả |
|----------|----------|
| `listing.parquet` (tuỳ chọn) | Nếu `use_listing_tickers=True`: lấy danh sách mã từ pipeline có cấu trúc |
| `sources.yaml` | Feed RSS + HTML spec |
| `.env` | `VNSTOCK_API_KEY` |

| Đầu ra | Đường dẫn |
|---------|----------|
| vnstock | `data-lake/raw/Unstructure_Data/news/vnstock/date=<ngày>/PART-000.parquet` |
| RSS | `data-lake/raw/Unstructure_Data/news/rss/date=<ngày>/PART-000.parquet` |
| HTML | `data-lake/raw/Unstructure_Data/news/html/date=<ngày>/PART-000.parquet` |
| Gộp (để tắt) | `data-lake/raw/Unstructure_Data/news/items/date=<ngày>/PART-000.parquet` |

In [ ]:
import os, sys, warnings, threading

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

_orig_hook = threading.excepthook
def _quiet_hook(args):
    if "UnicodeDecodeError" in str(args.exc_type):
        pass
    else:
        _orig_hook(args)
threading.excepthook = _quiet_hook
warnings.filterwarnings("ignore")

from pathlib import Path
root = Path.cwd()
if not (root / "ingestion").is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from ingestion.common import configure_logging, load_dotenv_from_project_root
from ingestion.unstructured_data import NewsIngestionConfig, ingest_news, primary_news_display_path

configure_logging()
load_dotenv_from_project_root()
print("[OK] setup")

## Cấu hình

In [ ]:
news_cfg = NewsIngestionConfig(
    use_listing_tickers=True,
    listing_exchange_filter=["HSX", "HNX"],
    max_tickers_per_run=50,
    max_articles_per_source=200,
    days_back=0,
    enable_vnstock=True,
    enable_rss=True,
    enable_html=True,
)

print(f"run_date      : {news_cfg.run_date}")
print(f"news_root     : {news_cfg.news_root}")
print(f"sources.yaml  : {news_cfg.resolved_sources_yaml()}")
print(f"listing       : {news_cfg.resolved_listing_parquet()}")

## Chạy

In [ ]:
news_paths = ingest_news(news_cfg)
news_paths

## Kiểm tra kết quả

In [ ]:
import pandas as pd

for key, path in news_paths.items():
    if not path:
        print(f"{key:10s}: (trống)")
        continue
    df = pd.read_parquet(path)
    print(f"{key:10s}: {len(df):>5d} dòng  →  {path}")

print(f"\nPrimary display path: {primary_news_display_path(news_paths)}")